# 02 — LoRA Rank Sweep on Banking77

Part of [bert-lora-finetuning](../README.md). Independent of `01_baseline...ipynb` — this notebook
loads its own data and only trains LoRA models, at several ranks. It doesn't need 01 to run, but the
accuracy plot is more useful with 01's full fine-tune number as a reference line.

**What this answers:** as rank `r` increases (more trainable parameters), how does accuracy change, and
where do returns start diminishing? This turns the single `r=8` number from notebook 01 into an actual
empirical curve instead of one point.

**Run on Colab or Kaggle with a GPU runtime.** This trains 4 separate LoRA models — expect roughly
4x the LoRA training time from notebook 01.


In [ ]:
!pip install -q transformers datasets peft accelerate evaluate torch scikit-learn


In [ ]:
import gc
import json
import os
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU detected — this notebook trains 4 models, CPU will take a long time.")

RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Paste the full-fine-tune accuracy printed at the end of 01_baseline_full_finetune_vs_lora.ipynb here.
# Leave as None to skip the reference line on the accuracy plot.
FULL_FINETUNE_ACCURACY = None  # e.g. 0.9123


In [ ]:
dataset = load_dataset("PolyAI/banking77")
label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(f"Number of intent classes: {num_labels}")

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_ds = tokenized["train"]
eval_ds = tokenized["test"]
print(f"Train size: {len(train_ds)}, Test size: {len(eval_ds)}")


In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    macro_f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": macro_f1}

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def peak_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return None


## Sweep: train LoRA at r = 4, 8, 16, 32

In [ ]:
RANKS = [4, 8, 16, 32]
ALPHA_TO_RANK_RATIO = 2  # keep alpha/r scaling constant (alpha = 2r) across the sweep

sweep_results = []

for r in RANKS:
    print(f"\n=== Training LoRA with r={r} ===")

    base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=r,
        lora_alpha=r * ALPHA_TO_RANK_RATIO,
        lora_dropout=0.1,
        target_modules=["query", "value"],
    )
    model = get_peft_model(base_model, lora_config)
    model.to(device)

    trainable, total = count_trainable_params(model)

    args = TrainingArguments(
        output_dir=f"./lora_sweep_r{r}",
        learning_rate=2e-4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=4,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
    )

    reset_peak_memory()
    start = time.time()
    trainer.train()
    train_time = time.time() - start
    peak_mem = peak_memory_mb()

    eval_results = trainer.evaluate()

    sweep_results.append({
        "rank": r,
        "alpha": r * ALPHA_TO_RANK_RATIO,
        "trainable_params": trainable,
        "trainable_pct": round(100 * trainable / total, 3),
        "accuracy": eval_results["eval_accuracy"],
        "macro_f1": eval_results["eval_macro_f1"],
        "train_time_s": round(train_time, 1),
        "peak_mem_mb": round(peak_mem, 1) if peak_mem else None,
    })

    print(f"r={r}: acc={eval_results['eval_accuracy']:.4f}, "
          f"trainable={trainable:,} ({100*trainable/total:.2f}%), time={train_time:.1f}s")

    # free memory before the next rank
    del model, base_model, trainer
    gc.collect()
    torch.cuda.empty_cache()

sweep_df = pd.DataFrame(sweep_results)
sweep_df


In [ ]:
sweep_df.to_csv(f"{RESULTS_DIR}/02_rank_sweep.csv", index=False)
with open(f"{RESULTS_DIR}/02_rank_sweep.json", "w") as f:
    json.dump(sweep_df.to_dict(orient="records"), f, indent=2)
print(f"Saved results to {RESULTS_DIR}/02_rank_sweep.{{csv,json}}")


## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(sweep_df["rank"], sweep_df["accuracy"], marker="o", color="#55A868", label="LoRA")
if FULL_FINETUNE_ACCURACY is not None:
    axes[0].axhline(FULL_FINETUNE_ACCURACY, color="#4C72B0", linestyle="--", label="Full fine-tune")
axes[0].set_xlabel("Rank (r)")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy vs. Rank")
axes[0].set_xscale("log", base=2)
axes[0].legend()

axes[1].plot(sweep_df["rank"], sweep_df["trainable_params"], marker="o", color="#C44E52")
axes[1].set_xlabel("Rank (r)")
axes[1].set_ylabel("Trainable Parameters")
axes[1].set_title("Trainable Parameters vs. Rank")
axes[1].set_xscale("log", base=2)
axes[1].set_yscale("log")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/02_rank_sweep.png", dpi=150)
plt.show()


**Read the curve, don't just eyeball it:** where does accuracy stop improving as `r` grows? That point
is the actual rank/performance trade-off — note it explicitly, with numbers, in the README.
